In [1]:
# Import library yang diperlukan

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

In [2]:
# Load dataset iris
iris = load_iris()


In [3]:
#dtype ndarray
x, y = iris.data, iris.target
print(type(x), type(y))

<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [4]:
# Menampilkan 5 data pertama dari x dan y dalam bentuk numpy array

print(x[:5])
print(y[:5])

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
[0 0 0 0 0]


In [5]:
# dtype pandas DataFrame
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [6]:
# Modeling dengan Scikit-learn
# Syarat Modeling: tidak ada null, tidak ada missing value, data sudah dalam bentuk matriks atau sudah berupa numerik.
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [7]:
# Preprocessing data simpel sekaligus modeling dengan Scikit-learn lewat pipeline
from re import S

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# Pipeline ndarray
pipe_numpy = Pipeline([("scaler", StandardScaler()), ("classifier", LinearSVC())])

# Pipeline pandas DataFrame
pipe_pandas = Pipeline([("scaler", StandardScaler()), ("classifier", LinearSVC())])


In [8]:
# Spliting data
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# split ndarray
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=46) 

# split pandas DataFrame
df_train, df_test = train_test_split(df, test_size=0.2, random_state=46) 

# Kenapa berbeda? Karena pada ndarray antara x sebagai data dan y sebagai target terpisah, sedangkan pada pandas DataFrame antara data dan target masih dalam satu dataframe yang sama. Sehingga pada saat spliting data, pada ndarray kita harus memisahkan antara data dan target terlebih dahulu, sedangkan pada pandas DataFrame kita bisa langsung spliting data tanpa harus memisahkan antara data dan target terlebih dahulu.

In [9]:
# Training
# Training dengan ndarray
pipe_numpy.fit(x_train, y_train)
y_pred_numpy = pipe_numpy.predict(x_test)
print('Hasil Evaluasi Model Numpy')
print(classification_report(y_test, y_pred_numpy))

Hasil Evaluasi Model Numpy
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.88      0.78      0.82         9
           2       0.80      0.89      0.84         9

    accuracy                           0.90        30
   macro avg       0.89      0.89      0.89        30
weighted avg       0.90      0.90      0.90        30



In [10]:
#  Training dengan pandas DataFrame
x_train_df, y_train_df = df_train.drop('target', axis=1), df_train['target']
x_test_df, y_test_df = df_test.drop('target', axis=1), df_test['target']
pipe_pandas.fit(x_train_df, y_train_df)
y_pred_pandas = pipe_pandas.predict(x_test_df)
print('Hasil Evaluasi Model Pandas DataFrame')
print(classification_report(y_test_df, y_pred_pandas))

Hasil Evaluasi Model Pandas DataFrame
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.88      0.78      0.82         9
           2       0.80      0.89      0.84         9

    accuracy                           0.90        30
   macro avg       0.89      0.89      0.89        30
weighted avg       0.90      0.90      0.90        30



In [12]:
# save model
import pickle

# Save model pipeline numpy
with open('model_numpy.pkl','wb') as model_numpy_file:
    pickle.dump(pipe_numpy, model_numpy_file)

In [14]:
# Save model pipeline pandas DataFrame
with open('model_pandas.pkl', 'wb') as model_pandas_file:
    pickle.dump(pipe_pandas, model_pandas_file)

In [16]:
# Load model pipeline numpy
with open('../models/model_numpy.pkl', 'rb') as model_numpy_file:
    loaded_numpy_model = pickle.load(model_numpy_file)


In [ ]:
# coba predict
new_data = [[1,1,1,1]] # tidak bisa menggunakan vektor [1,1,1,1], harus berupa matriks [[1,1,1,1]]
loaded_numpy_model.predict(new_data) # bisa saja isi variabeel new_data diubah menjadi vektor [1,1,1,1] tapi di .predict harus ada [] contoh: loaded_numpy_model.predict([new_data]).


array([1])

In [ ]:
# apa artinya array([1])? ini bisa diartikan bahwa model memprediksi bahwa data baru yang kita masukkan termasuk ke dalam kelas 1. Kelas 1 ini merujuk pada target yang ada pada dataset iris, yaitu 'versicolor'. Jadi, model memprediksi bahwa data baru yang kita masukkan termasuk ke dalam kelas 'versicolor'.
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [19]:
# load model pipeline pandas DataFrame
with open('../models/model_pandas.pkl', 'rb') as model_pandas_file:
    loaded_pandas_model = pickle.load(model_pandas_file)

In [27]:
#  coba predict dengan model pandas DataFrame
# catatan : karena model pandas DataFrame dilatih dengan data yang masih berupa DataFrame, maka saat predict juga harus menggunakan DataFrame yang tentunya memiliki kolom, tidak bisa menggunakan list atau array. Jadi, kita harus membuat DataFrame baru yang memiliki kolom yang sama dengan DataFrame yang digunakan saat training, yaitu 'sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'. Contoh:
new_data_df = pd.DataFrame(data=[[3,3,3,3]], columns=iris.feature_names)
new_data_df


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,3,3,3,3


In [ ]:
loaded_pandas_model.predict(new_data_df)

# apa artinya array([2])? ini bisa diartikan bahwa model memprediksi bahwa data baru yang kita masukkan termasuk ke dalam kelas 2. Kelas 2 ini merujuk pada target yang ada pada dataset iris, yaitu 'virginica'. Jadi, model memprediksi bahwa data baru yang kita masukkan termasuk ke dalam kelas 'virginica'.

array([2])

In [29]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')